In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog_name = 'ecommerce'

In [0]:
df_silver = spark.table(f'{catalog_name}.silver.slv_orders_items')

display(df_silver.limit(5))

In [0]:
# add gross amount

df_gold = df_silver.withColumn('gross_amount',
                               col('quantity')*col('unit_price'))

# add discount_amount

df_gold = df_gold.withColumn('discount_amount', ceil(col('gross_amount')*col('discount_pct')/100.0))     

# add sale_amount = gross - discount + tax_amount

df_gold = df_gold.withColumn('sale_amount', col('gross_amount') - col('discount_amount') + col('tax_amount'))

# add date id

df_gold = df_gold.withColumn('date_id', date_format(col('dt'),'yyyyMMdd').cast(IntegerType()))

# add coupon flag

df_gold = df_gold.withColumn('coupon_flag',
                             when(col('coupon_code').isNotNull(), lit(1))
                             .otherwise(lit(0)))

In [0]:
# fix rate

fx_rates = {
    'INR': 1.00,
    'AED': 24.18,
    'AUD': 57.55,
    'CAD': 62.93,
    'GBP': 117.98,
    'SGD': 68.18,
    'USD': 88.29,
}

rates = [(k, float(v)) for k, v in fx_rates.items()]
rates_df = spark.createDataFrame(rates, ['currency','inr_rate'])
rates_d'show()

In [0]:
df_gold = (
    df_gold
    .join(
        rates_df,
        rates_d'currency == upper(trim(col('unit_price_currency'))),
        'left'
    )
    .withColumn('sale_amount_inr', col('sale_amount') * col('inr_rate'))
    .withColumn('sale_amount_inr', ceil(col('sale_amount_inr')))
)

In [0]:
display(df_gold.limit(5))

In [0]:
df_gold.printSchema()

In [0]:
df_gold = df_gold.select(
     col('date_id'),
     col('dt').alias('transaction_date'),
     col('order_ts').alias('transaction_ts'),
     col('order_id').alias('transaction_id'),
     col('customer_id'),
     col('item_seq').alias('seq_no'),
     col('product_id'),
     col('channel'),
     col('coupon_code'),
     col('coupon_flag'),
     col('unit_price_currency'),
     col('quantity'),
     col('unit_price'),
     col('gross_amount'),
     col('discount_pct').alias('discount_percent'),
     col('discount_amount'),
     col('tax_amount'),
     col('sale_amount').alias('net_amount'),
     col('sale_amount_inr').alias('net_amount_inr')
)

In [0]:
display(df_gold.limit(5))

In [0]:
# create the final table in the gold layer

df_gold.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.gold.gld_orders_items')